<div dir="rtl">

# ۳ – ارزیابی ساده‌ی RAG و جمع‌بندی نتایج

در این نوت‌بوک، مجموعه‌ای از سؤال‌های ارزیابی را آماده می‌کنیم، سیستم RAG خود را روی آن‌ها اجرا می‌کنیم، کیفیت پاسخ‌ها را بررسی می‌کنیم و در پایان نتایج را به‌صورت کوتاه جمع‌بندی می‌کنیم.

</div>


In [ ]:

import pandas as pd
import numpy as np
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer
import faiss
import pickle
from typing import List, Dict

In [ ]:
main_dir = "Desktop/BOOTCAMP/S9-RAG/پروژه  دوره RAG/rag-persian-news-lite-main"

chunks_df = pd.read_csv(f'{main_dir}/data/processed/chunks.csv')
semantic_index = faiss.read_index(f'{main_dir}/data/processed/semantic_index.faiss')
lexical_index = faiss.read_index(f'{main_dir}/data/processed/lexical_index.faiss')
hybrid_index = faiss.read_index(f'{main_dir}/data/processed/hybrid_index.faiss')
with open(f'{main_dir}/data/processed/tfidf_vectorizer.pkl', 'rb') as f:
    tfidf_vectorizer = pickle.load(f)
print("     TF-IDF vectorizer loaded")

print(" Loading alpha value...")
with open(f'{main_dir}/data/processed/alpha.txt', 'r') as f:
    alpha = float(f.read().strip())
print(f"    Alpha = {alpha}")
semantic_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')


In [ ]:
evaluation_questions = [
    {
        "id": "q1",
        "question": "آخرین اخبار فوتبال چیست؟",
        "reference_answer": "اخبار مربوط به مسابقات فوتبال، نتایج بازی‌ها، انتقالات بازیکنان و اتفاقات مهم فوتبالی",
        "doc_ids": []
    },
    {
        "id": "q2",
        "question": "در مورد وضعیت اقتصادی ایران چه اخباری هست؟",
        "reference_answer": "اخبار مربوط به اقتصاد، تجارت، بازار، قیمت‌ها و مسائل مالی ایران",
        "doc_ids": []
    },
    {
        "id": "q3",
        "question": "اخبار سیاسی امروز چیست؟",
        "reference_answer": "اخبار مربوط به رویدادهای سیاسی، تصمیمات دولتی و مسائل سیاسی داخلی و خارجی",
        "doc_ids": []
    },
    {
        "id": "q4",
        "question": "در مورد تکنولوژی و فناوری چه خبر؟",
        "reference_answer": "اخبار مربوط به پیشرفت‌های تکنولوژیکی، نوآوری‌های فنی و محصولات جدید فناوری",
        "doc_ids": []
    },
    {
        "id": "q5",
        "question": "آخرین اخبار ورزشی کشور چیست؟",
        "reference_answer": "اخبار مربوط به رویدادهای ورزشی، ورزشکاران، مسابقات و دستاوردهای ورزشی",
        "doc_ids": []
    },
    {
        "id": "q6",
        "question": "وضعیت آب و هوا و محیط زیست چطور است؟",
        "reference_answer": "اخبار مربوط به شرایط آب و هوایی، محیط زیست، آلودگی و تغییرات اقلیمی",
        "doc_ids": []
    },
    {
        "id": "q7",
        "question": "در حوزه فرهنگ و هنر چه اتفاقی افتاده؟",
        "reference_answer": "اخبار مربوط به رویدادهای فرهنگی، هنری، سینما، موسیقی و ادبیات",
        "doc_ids": []
    },
    {
        "id": "q8",
        "question": "اخبار بهداشت و سلامت چیست؟",
        "reference_answer": "اخبار مربوط به بهداشت عمومی، بیماری‌ها، درمان و پیشرفت‌های پزشکی",
        "doc_ids": []
    },
    {
        "id": "q9",
        "question": "در مورد آموزش و پرورش چه خبری هست؟",
        "reference_answer": "اخبار مربوط به سیستم آموزشی، مدارس، دانشگاه‌ها و تحصیلات",
        "doc_ids": []
    },
    {
        "id": "q10",
        "question": "آخرین اخبار بین‌المللی چیست؟",
        "reference_answer": "اخبار مربوط به رویدادهای جهانی، سیاست خارجی و اتفاقات بین‌المللی",
        "doc_ids": []
    },
    {
        "id": "q11",
        "question": "وضعیت حمل و نقل و ترافیک چطور است؟",
        "reference_answer": "اخبار مربوط به حمل و نقل عمومی، ترافیک شهری، راه‌ها و مسافرت",
        "doc_ids": []
    },
    {
        "id": "q12",
        "question": "اخبار علمی و تحقیقاتی چیست؟",
        "reference_answer": "اخبار مربوط به تحقیقات علمی، کشفیات جدید و دستاوردهای پژوهشی",
        "doc_ids": []
    },
    {
        "id": "q13",
        "question": "در مورد صنعت و تولید چه خبر؟",
        "reference_answer": "اخبار مربوط به صنایع مختلف، کارخانجات، تولید و صادرات",
        "doc_ids": []
    },
    {
        "id": "q14",
        "question": "وضعیت بازار و قیمت‌ها چطور است؟",
        "reference_answer": "اخبار مربوط به بازار کالاها، قیمت‌ها، عرضه و تقاضا",
        "doc_ids": []
    },
    {
        "id": "q15",
        "question": "اخبار اجتماعی و شهری چیست؟",
        "reference_answer": "اخبار مربوط به مسائل اجتماعی، رویدادهای شهری و زندگی روزمره",
        "doc_ids": []
    }
]

# Save questions to JSONL  
questions_file = f'{main_dir}/evaluation/questions_template.jsonl'
print(f"\n Saving {len(evaluation_questions)} questions to {questions_file}")

with open(questions_file, 'w', encoding='utf-8') as f:
    for q in evaluation_questions:
        f.write(json.dumps(q, ensure_ascii=False) + '\n')

print(f"saved successfully")


In [ ]:

def load_questions_from_jsonl(filepath: str) -> List[Dict]:
 
    questions = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                questions.append(json.loads(line))
    return questions
 
loaded_questions = load_questions_from_jsonl(questions_file)
print(f"   ✅ Loaded {len(loaded_questions)} questions")

for q in loaded_questions[:3]:
    print(f"   - [{q['id']}] {q['question']}")
 
def retrieve(question: str, embedding_model: str = 'hybrid', top_k: int = 5) -> List[Dict]:
     
    if embedding_model == 'semantic':
        query_embedding = semantic_model.encode([question], normalize_embeddings=True)
        index = semantic_index
        
    elif embedding_model == 'lexical':
        query_vec = tfidf_vectorizer.transform([question]).toarray()
        query_norm = np.linalg.norm(query_vec)
        query_embedding = query_vec / (query_norm + 1e-10)
        index = lexical_index
        
    else:  # hybrid
        sem_emb = semantic_model.encode([question], normalize_embeddings=True)
        lex_vec = tfidf_vectorizer.transform([question]).toarray()
        lex_norm = np.linalg.norm(lex_vec)
        lex_emb = lex_vec / (lex_norm + 1e-10)
        query_embedding = alpha * sem_emb + (1 - alpha) * lex_emb
        query_norm = np.linalg.norm(query_embedding)
        query_embedding = query_embedding / (query_norm + 1e-10)
        index = hybrid_index
     
    scores, indices = index.search(query_embedding.astype('float32'), top_k)
     
    results = []
    for idx, score in zip(indices[0], scores[0]):
        chunk_info = chunks_df.iloc[idx]
        results.append({
            'chunk_idx': int(idx),
            'score': float(score),
            'chunk_id': chunk_info['chunk_id'],
            'doc_id': chunk_info['doc_id'],
            'title': chunk_info['title'],
            'category': chunk_info['category'],
            'date': chunk_info['date'],
            'text': chunk_info['text']
        })
    
    return results

def answer(question: str, embedding_model: str = 'hybrid', top_k: int = 5) -> Dict:
    """Generate answer using RAG"""
    retrieved = retrieve(question, embedding_model, top_k)
     
    context_parts = []
    for i, chunk in enumerate(retrieved, 1):
        context_parts.append(f"{chunk['text']}")
    
    context = "\n\n".join(context_parts)
     
    simple_answer = "\n\n".join([c['text'][:400] for c in retrieved[:3]])
    
    return {
        'question': question,
        'embedding_model': embedding_model,
        'retrieved_chunks': retrieved,
        'context': context,
        'answer': simple_answer
    }
 


In [ ]:
all_results = []

for q_data in evaluation_questions:
    question = q_data['question']
    q_id = q_data['id']
    
    print(f"\n{'='*70}")
    print(f"[{q_id}] {question}")
    print(f"{'='*70}")
    
    result_entry = {
        'question_id': q_id,
        'question': question,
        'reference_answer': q_data['reference_answer']
    }
    
    for emb_type in ['semantic', 'lexical', 'hybrid']:
        retrieved = retrieve(question, embedding_model=emb_type, top_k=5)
        answer_result = answer(question, embedding_model=emb_type, top_k=5)
        
        result_entry[f'{emb_type}_retrieved'] = retrieved
        result_entry[f'{emb_type}_answer'] = answer_result['answer']
        result_entry[f'{emb_type}_top_score'] = retrieved[0]['score'] if retrieved else 0
        
        print(f"\n  {emb_type.upper():10s} | Top-5 Results:")
        for i, chunk in enumerate(retrieved, 1):
            print(f"    {i}. [Score: {chunk['score']:.4f}] {chunk['title'][:60]}...")
            print(f"       Category: {chunk['category']:15s} | {chunk['text'][:100]}...")
    
    all_results.append(result_entry)


In [ ]:

print("\n\n" + "="*70)
print("STEP 5: SIMPLE AUTOMATIC EVALUATION")
print("="*70)

def evaluate_results(results: List[Dict], score_threshold: float = 0.25) -> Dict:
    """
    Simple evaluation based on retrieval scores
    
    Args:
        results: List of evaluation results
        score_threshold: Minimum score to consider as acceptable
    
    Returns:
        Statistics dictionary
    """
    stats = {
        'semantic': {'accept': 0, 'reject': 0, 'scores': []},
        'lexical': {'accept': 0, 'reject': 0, 'scores': []},
        'hybrid': {'accept': 0, 'reject': 0, 'scores': []}
    }
    
    for result in results:
        for emb_type in ['semantic', 'lexical', 'hybrid']:
            top_score = result[f'{emb_type}_top_score']
            stats[emb_type]['scores'].append(top_score)
            
            # Accept if top score is above threshold
            if top_score >= score_threshold:
                stats[emb_type]['accept'] += 1
            else:
                stats[emb_type]['reject'] += 1
    
    # Calculate averages
    for emb_type in ['semantic', 'lexical', 'hybrid']:
        stats[emb_type]['avg_score'] = np.mean(stats[emb_type]['scores'])
        stats[emb_type]['median_score'] = np.median(stats[emb_type]['scores'])
    
    return stats

# Evaluate
print("\n📊 Evaluating with score threshold = 0.25...")
evaluation_stats = evaluate_results(all_results, score_threshold=0.25)

# Display results
print("\n" + "="*70)
print("📈 EVALUATION RESULTS")
print("="*70)

for emb_type in ['semantic', 'lexical', 'hybrid']:
    stats = evaluation_stats[emb_type]
    total = stats['accept'] + stats['reject']
    accuracy = (stats['accept'] / total * 100) if total > 0 else 0
    
    print(f"\n{emb_type.upper()}:")
    print(f"  ✓ Acceptable answers: {stats['accept']}/{total} ({accuracy:.1f}%)")
    print(f"  ✗ Rejected answers:   {stats['reject']}/{total} ({100-accuracy:.1f}%)")
    print(f"  📊 Average score:     {stats['avg_score']:.4f}")
    print(f"  📊 Median score:      {stats['median_score']:.4f}")

# ============================================
# TODO: ۲–۳ نمونه پاسخ خوب و ۲–۳ نمونه پاسخ بد را چاپ کنید و 
# به صورت متنی توضیح دهید چرا خوب/بد هستند
# ============================================

print("\n\n" + "="*70)
print("STEP 6: SHOW GOOD AND BAD EXAMPLES")
print("="*70)

# Sort results by hybrid score to find good and bad examples
sorted_results = sorted(all_results, key=lambda x: x['hybrid_top_score'], reverse=True)

print("\n\n✨ GOOD EXAMPLES (High Retrieval Scores)")
print("="*70)

for i, result in enumerate(sorted_results[:3], 1):
    print(f"\n{'─'*70}")
    print(f"GOOD EXAMPLE {i}")
    print(f"{'─'*70}")
    print(f"\n❓ سوال: {result['question']}")
    print(f"📝 پاسخ مرجع: {result['reference_answer']}")
    
    for emb_type in ['semantic', 'lexical', 'hybrid']:
        score = result[f'{emb_type}_top_score']
        print(f"\n  [{emb_type.upper()}] امتیاز: {score:.4f}")
        
        top_chunk = result[f'{emb_type}_retrieved'][0]
        print(f"    عنوان: {top_chunk['title']}")
        print(f"    دسته: {top_chunk['category']}")
        print(f"    متن: {top_chunk['text'][:200]}...")
    

In [ ]:

comparison_data = []
for result in all_results:
    comparison_data.append({
        'question_id': result['question_id'],
        'question': result['question'][:50] + '...',
        'semantic_score': result['semantic_top_score'],
        'lexical_score': result['lexical_top_score'],
        'hybrid_score': result['hybrid_top_score']
    })

comparison_df = pd.DataFrame(comparison_data)

print("\n📊 Score Comparison Table:")
print(comparison_df.to_string(index=False))

print("\n📈 Statistical Summary:")
print(comparison_df[['semantic_score', 'lexical_score', 'hybrid_score']].describe())


<div dir="rtl">

## خلاصه برای گزارش
گزارش کامل در فایل دوم به دستتان خواهد رسید
</div>
